<a href="https://colab.research.google.com/github/group6aiml-hub/NIDS_AInML/blob/main/notebooks/02_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#1-2 Oppsett og Last inn data
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

path = '/content/drive/MyDrive/Datasets/'
train = pd.read_csv(path + 'UNSW_NB15_training-set.csv', encoding='utf-8-sig')
test  = pd.read_csv(path + 'UNSW_NB15_testing-set.csv',  encoding='utf-8-sig')
print(train.shape, test.shape)

Mounted at /content/drive
(175341, 45) (82332, 45)


In [2]:
#3 Fjern ID
train = train.drop(columns=['id'])
test  = test.drop(columns=['id'])

In [3]:
# 4. Manglende verdier
print(train.isna().sum().sum(), test.isna().sum().sum())

0 0


In [4]:
#5. Duplikater
dup = train.duplicated()
print(dup.sum(), round(dup.mean() * 100, 1), '%')
print(train.loc[dup, 'attack_cat'].value_counts())
train = train.drop_duplicates()
print(train.shape)

67601 38.6 %
attack_cat
Generic           35819
Exploits          13549
DoS                8458
Normal             4110
Reconnaissance     2969
Fuzzers            2034
Analysis            406
Backdoor            211
Shellcode            42
Worms                 3
Name: count, dtype: int64
(107740, 44)


In [5]:
#6 Motstridende rader
feats = train.columns.drop(['label', 'attack_cat']).tolist()
konflikt = train.groupby(feats)['attack_cat'].nunique()
print((konflikt > 1).sum())

1772


In [6]:
# 6b. Andel rader per klasse som ligger i en konfliktgruppe
# (samme features, men ulik attack_cat)
ant_fasiter = train.groupby(feats)['attack_cat'].transform('nunique')
i_konflikt  = ant_fasiter > 1

andel = i_konflikt.groupby(train['attack_cat']).mean().sort_values(ascending=False)
print((andel * 100).round(1))
print('Totalt:', round(i_konflikt.mean() * 100, 1), '% av radene')

attack_cat
Analysis          75.3
Backdoor          73.7
DoS               34.9
Reconnaissance    17.5
Fuzzers            9.6
Worms              7.9
Exploits           7.2
Generic            6.6
Shellcode          0.5
Normal             0.4
Name: attack_cat, dtype: float64
Totalt: 7.9 % av radene


**Contradictory labels per class:** 7.9 % of all rows share identical features with a row that has a different attack_cat. This is concentrated in a few classes: Analysis (75.3 %), Backdoor (73.7 %) and DoS (34.9 %). For these rows no model can be correct for both labels, which puts an upper limit on multiclass performance. Normal is almost unaffected (0.4 %), so the conflicts are mainly *between attack types*, which explains why binary performance (~0.93) is much higher than multiclass (~0.6).

In [7]:
# 7. Redundant FTP-kolonne
print((train['is_ftp_login'] == train['ct_ftp_cmd']).all())
train = train.drop(columns=['is_ftp_login'])
test  = test.drop(columns=['is_ftp_login'])

True


In [8]:
#8. Negative verdier
print((train.select_dtypes('number') < 0).sum().sum())

0


In [9]:
# 9. Klassefordeling etter rensing
print(train['attack_cat'].value_counts())

attack_cat
Normal            51890
Exploits          19844
Fuzzers           16150
Reconnaissance     7522
Generic            4181
DoS                3806
Analysis           1594
Backdoor           1535
Shellcode          1091
Worms               127
Name: count, dtype: int64


In [10]:
# 10. Grupper sjeldne kategorier
min_antall = 100
for c in ['proto', 'service', 'state']:
    tellinger = train[c].value_counts()
    behold = tellinger[tellinger >= min_antall].index
    train[c] = train[c].where(train[c].isin(behold), 'other')
    test[c]  = test[c].where(test[c].isin(behold), 'other')
    print(c, '→', sorted(behold.tolist()) + ['other'])

proto → ['arp', 'ospf', 'sctp', 'tcp', 'udp', 'unas', 'other']
service → ['-', 'dns', 'ftp', 'ftp-data', 'http', 'pop3', 'smtp', 'ssh', 'other']
state → ['CON', 'FIN', 'INT', 'REQ', 'other']


In [11]:
# 11. Kontrollsjekk
for c in ['proto', 'service', 'state']:
    print(c, '| bare i test:', set(test[c]) - set(train[c]))
print(train.shape, test.shape)

proto | bare i test: set()
service | bare i test: set()
state | bare i test: set()
(107740, 43) (82332, 43)


In [12]:
# 12. Lagre
train.to_csv(path + 'train_clean.csv', index=False)
test.to_csv(path + 'test_clean.csv',  index=False)
print('Lagret!')

Lagret!


In [13]:
# 13. Kontroll av testsettet (ingen modell brukes her)
rå_test = pd.read_csv(path + 'UNSW_NB15_testing-set.csv', encoding='utf-8-sig')

# a) Negative verdier
print('Negative verdier i test:', (test.select_dtypes('number') < 0).sum().sum())

# b) Er FTP-kolonnene like også i test?
print('is_ftp_login == ct_ftp_cmd i test:',
      (rå_test['is_ftp_login'] == rå_test['ct_ftp_cmd']).all())

# c) Duplikater i test (beholdes bevisst)
print('Duplikater i test:', test.duplicated().sum(),
      f'({test.duplicated().mean()*100:.1f} %)')

# d) Overlapp: testrader som finnes identisk i train
f = test.columns.drop(['label', 'attack_cat']).tolist()
i_train = test[f].merge(train[f].drop_duplicates(), on=f, how='left', indicator=True)['_merge'] == 'both'
print(f'Testrader som også finnes i train: {i_train.sum()} ({i_train.mean()*100:.1f} %)')
print((i_train.groupby(test['attack_cat'].values).mean() * 100).round(1).sort_values(ascending=False))

Negative verdier i test: 0
is_ftp_login == ct_ftp_cmd i test: False
Duplikater i test: 26621 (32.3 %)
Testrader som også finnes i train: 8618 (10.5 %)
DoS               49.8
Reconnaissance    32.7
Analysis          19.1
Exploits          16.4
Generic            9.1
Fuzzers            4.4
Normal             4.0
Shellcode          3.4
Backdoor           1.4
Worms              0.0
Name: _merge, dtype: float64


**Test set checks:** No negative values, so log1p is safe. `is_ftp_login` and `ct_ftp_cmd` are identical in train but not in test; the column was dropped based on train only, which is the correct procedure. The test set keeps its duplicates (32.3 %) on purpose, since real traffic repeats. 10.5 % of test rows also appear identically in train, concentrated in DoS (49.8 %) and Reconnaissance (32.7 %). Test results for these classes are therefore optimistic; in 08 we report macro-F1 both on the full test set and on the rows not seen in train.